In [19]:
# ============================================================
# TÜRK MARKET — SATIŞ TAHMİN MODELİ v4 (t+1, LEAKAGE YOK)
#
# PROBLEM TANIMI:
#   Bugünün (t) bilgisiyle yarının (t+1) satışını tahmin et.
#   "Şube X'te kategori Y yarın kaç birim satılacak?"
#
# LEAKAGE ÖNLEMİ — KURALLAR:
#   1. Tahmin günü t için sadece t-1 ve öncesi kullanılır
#   2. unique_clients, female_ratio, unique_items → O GÜNÜN
#      verisi olduğu için KALDIRILDI
#   3. cat_avg_price → split'ten SONRA sadece train'den hesaplanır
#   4. Tüm lag/rolling feature'lar shift(1) ile geçmişe bakıyor
#   5. Hedef sütunu shift(-1) ile bir gün ileri kaydırıldı (yarın)
#
# AGGREGATE:
#   DATE + BRANCH + CATEGORY_NAME1 → günlük toplam satış
# ============================================================
 
import os
import warnings
import numpy as np
import pandas as pd
import kagglehub
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import h2o
from h2o.automl import H2OAutoML
 
warnings.filterwarnings("ignore")
 
try:
    os.chdir(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    pass
 
os.makedirs("results", exist_ok=True)
os.makedirs("results/plots", exist_ok=True)
 
print("=" * 60)
print("Türk Market — t+1 Satış Tahmin Modeli (Leakage Yok)")
print("=" * 60)
 
# ============================================================
# 1️⃣ VERİYİ İNDİR VE YÜKLE
# ============================================================
 
print("\n[1/7] Veri indiriliyor...")
 
dataset_path = kagglehub.dataset_download(
    "omercolakoglu/turkish-market-sales-dataset-with-9000items"
)
 
excel_files = []
for root, dirs, files in os.walk(dataset_path):
    for file in files:
        if file.lower().endswith(('.xlsx', '.xls')):
            excel_files.append(os.path.join(root, file))
 
dfs = []
for f in excel_files:
    try:
        _df = pd.read_excel(f)
        _df.columns = _df.columns.str.strip()
        dfs.append(_df)
    except Exception as e:
        print(f"  ⚠️  Atlandı: {e}")
 
df = pd.concat(dfs, ignore_index=True)
df.columns = df.columns.str.strip()
print(f"Ham satır: {len(df):,}")


Türk Market — t+1 Satış Tahmin Modeli (Leakage Yok)

[1/7] Veri indiriliyor...
Ham satır: 611,108


In [20]:
# ============================================================
# 2️⃣ TEMİZLİK
# ============================================================
 
print("\n[2/7] Temizlik...")
 
df['DATE_'] = pd.to_datetime(df['DATE_'], errors='coerce')
df = df.dropna(subset=['DATE_', 'AMOUNT', 'ITEMNAME', 'BRANCH'])
df = df[df['AMOUNT'] > 0]
 
noise = r"POSET|POŞET|KARGO|İNDİRİM|INDIRIM|TEST|İADE|IADE|HİZMET|HIZMET"
df = df[~df['ITEMNAME'].str.contains(noise, case=False, na=False, regex=True)]
 
for col in ['CATEGORY_NAME1', 'CATEGORY_NAME2', 'BRANCH', 'REGION']:
    if col in df.columns:
        df[col] = df[col].fillna('Bilinmiyor').astype(str).str.strip().str.upper()
 
print(f"Tarih aralığı : {df['DATE_'].min().date()} → {df['DATE_'].max().date()}")
print(f"Temiz satır   : {len(df):,}")
 
# ============================================================
# 3️⃣ AGGREGATE — ŞUBE + KATEGORİ + GÜN
# ============================================================
 
print("\n[3/7] Aggregate hesaplanıyor...")
 
daily = (
    df.groupby(['DATE_', 'BRANCH', 'CATEGORY_NAME1'])
    .agg(
        AMOUNT=('AMOUNT', 'sum'),    # günlük toplam satış — TAHMİN HEDEFİ
        PRICE=('PRICE', 'mean'),     # ortalama fiyat — geçmişe ait, leakage yok
    )
    .reset_index()
    .rename(columns={'DATE_': 'date'})
    .sort_values(['BRANCH', 'CATEGORY_NAME1', 'date'])
    .reset_index(drop=True)
)
 
# REGION bilgisini ekle (şube bazında sabit, leakage yok)
if 'REGION' in df.columns:
    region_map = df.groupby('BRANCH')['REGION'].first()
    daily['REGION'] = daily['BRANCH'].map(region_map).fillna('Bilinmiyor')
 
# Nadir kombinasyonları filtrele
combo_counts = daily.groupby(['BRANCH', 'CATEGORY_NAME1']).size()
valid_combos = combo_counts[combo_counts >= 14].index
daily = daily.set_index(['BRANCH', 'CATEGORY_NAME1'])
daily = daily[daily.index.isin(valid_combos)].reset_index()
 
print(f"Aggregate satır    : {len(daily):,}")
print(f"Benzersiz kombo    : {daily.groupby(['BRANCH','CATEGORY_NAME1']).ngroups:,}")
print(f"AMOUNT ort/std     : {daily['AMOUNT'].mean():.1f} / {daily['AMOUNT'].std():.1f}")
 
# ============================================================
# 4️⃣ HEDEF: t+1 (YARIN)
#
# shift(-1) ile hedefi bir gün ileri kaydırıyoruz.
# Yani bugünün feature'ları → yarının satışını tahmin eder.
# Son günün hedefi NaN olur → o satır silinir.
# ============================================================
 
print("\n[4/7] t+1 hedef oluşturuluyor...")
 
grp_target = daily.groupby(['BRANCH', 'CATEGORY_NAME1'])['AMOUNT']
daily['target_tomorrow'] = grp_target.shift(-1)  # yarının satışı
 
TARGET = 'target_tomorrow'
 
# Son gün için target NaN — bu satırları sil
before = len(daily)
daily = daily.dropna(subset=[TARGET])
print(f"Son gün satırları silindi: {before - len(daily)} satır")
 




[2/7] Temizlik...
Tarih aralığı : 2017-01-02 → 2017-03-31
Temiz satır   : 595,723

[3/7] Aggregate hesaplanıyor...
Aggregate satır    : 53,599
Benzersiz kombo    : 855
AMOUNT ort/std     : 14.2 / 37.5

[4/7] t+1 hedef oluşturuluyor...
Son gün satırları silindi: 855 satır


In [21]:
# ============================================================
# 5️⃣ FEATURE ENGINEERING (SADECE GEÇMİŞ VERİ)
#
# LEAKAGE ÖNLEMİ:
#   Tüm feature'lar bugünün (t) veya öncesinin verisi.
#   unique_clients, female_ratio → o günün verisi olduğu için YOK.
#   Fiyat → geçmişe ait, hedeften türetilmiyor → GÜVENLİ.
# ============================================================
 
print("\n[5/7] Feature engineering (sadece t ve öncesi)...")
 
# Tarih öznitelikleri (bugün için — leakage yok)
daily['dayofweek']    = daily['date'].dt.dayofweek.astype(int)
daily['month']        = daily['date'].dt.month.astype(int)
daily['weekofyear']   = daily['date'].dt.isocalendar().week.astype(int)
daily['is_weekend']   = (daily['date'].dt.dayofweek >= 5).astype(int)
daily['day']          = daily['date'].dt.day.astype(int)
daily['is_monday']    = (daily['date'].dt.dayofweek == 0).astype(int)
daily['is_friday']    = (daily['date'].dt.dayofweek == 4).astype(int)
 
# Ayın başı/ortası/sonu
daily['month_period'] = pd.cut(
    daily['date'].dt.day,
    bins=[0, 10, 20, 31],
    labels=['bas', 'orta', 'son']
).astype(str)
 
# Lag feature'lar — bugünün (t) ve geçmişin satışları
# shift(0) = bugün, shift(1) = dün, shift(7) = 1 hafta önce
# Hepsi t+1 için geçerli feature (leakage yok)
grp = daily.groupby(['BRANCH', 'CATEGORY_NAME1'])['AMOUNT']
 
daily['lag_0']  = grp.shift(0)   # bugün (tahmin anında bilinir)
daily['lag_1']  = grp.shift(1)   # dün
daily['lag_2']  = grp.shift(2)   # 2 gün önce
daily['lag_7']  = grp.shift(7)   # 1 hafta önce (aynı gün)
daily['lag_14'] = grp.shift(14)  # 2 hafta önce (aynı gün)
 
# Hareketli ortalamalar (bugün dahil, geçmişe bakıyor)
daily['rolling_mean_3']  = grp.transform(
    lambda x: x.rolling(3,  min_periods=1).mean())
daily['rolling_mean_7']  = grp.transform(
    lambda x: x.rolling(7,  min_periods=1).mean())
daily['rolling_mean_14'] = grp.transform(
    lambda x: x.rolling(14, min_periods=1).mean())
 
# Hareketli std
daily['rolling_std_7'] = grp.transform(
    lambda x: x.rolling(7, min_periods=2).std().fillna(0))
 
# Haftalık trend
daily['weekly_trend'] = daily['rolling_mean_7'] - daily['rolling_mean_14']
 
# Yarın hafta sonu mu? (hafta sonu satışları farklı olabilir)
tomorrow = daily['date'] + pd.Timedelta(days=1)
daily['tomorrow_is_weekend'] = (tomorrow.dt.dayofweek >= 5).astype(int)
daily['tomorrow_dayofweek']  = tomorrow.dt.dayofweek.astype(int)
 
print(f"  Feature'lar eklendi. Toplam sütun: {len(daily.columns)}")
 
# ============================================================
# 6️⃣ KRONOLOJİK SPLIT
#
# LEAKAGE ÖNLEMİ:
#   Rastgele split YAPILMAZ.
#   cat_avg_price SADECE train'den hesaplanır.
#   Test seti hiçbir hesaba kaynak olmaz.
# ============================================================
 
print("\n[6/7] Kronolojik split...")
 
max_date   = daily['date'].max()
split_date = max_date - pd.Timedelta(days=14)
 
train = daily[daily['date'] <= split_date].copy()
test  = daily[daily['date'] >  split_date].copy()
 
# cat_avg_price: kategorinin ortalama fiyatı
# SADECE train'den hesaplanır — test dahil edilmez
cat_price_avg = train.groupby('CATEGORY_NAME1')['PRICE'].mean()
train['cat_avg_price'] = train['CATEGORY_NAME1'].map(cat_price_avg)
test['cat_avg_price']  = test['CATEGORY_NAME1'].map(cat_price_avg).fillna(
    train['PRICE'].mean()
)
train['price_vs_cat'] = train['PRICE'] - train['cat_avg_price']
test['price_vs_cat']  = test['PRICE']  - test['cat_avg_price']
 
# Lag olmayan satırları sadece train'den kaldır
train = train.dropna(subset=['lag_1', 'lag_7', 'rolling_mean_7'])
 
# Sayısal null'ları train medyanıyla doldur
num_cols = train.select_dtypes(include='number').columns.tolist()
for col in num_cols:
    if col != TARGET and train[col].isnull().sum() > 0:
        med = train[col].median()
        train[col] = train[col].fillna(med)
        test[col]  = test[col].fillna(med)
 
print(f"Split tarihi   : {split_date.date()}")
print(f"Train          : {len(train):,} ({train['date'].min().date()} → {train['date'].max().date()})")
print(f"Test           : {len(test):,}  ({test['date'].min().date()} → {test['date'].max().date()})")
print(f"Train/Test oran: {len(train)/len(test):.1f}x")
 
# Log transform
train['log_target'] = np.log1p(train[TARGET])
test['log_target']  = np.log1p(test[TARGET])
LOG_TARGET = 'log_target'
 



[5/7] Feature engineering (sadece t ve öncesi)...
  Feature'lar eklendi. Toplam sütun: 27

[6/7] Kronolojik split...
Split tarihi   : 2017-03-16
Train          : 38,249 (2017-01-09 → 2017-03-16)
Test           : 8,510  (2017-03-17 → 2017-03-30)
Train/Test oran: 4.5x


In [22]:
# ============================================================
# 7️⃣ H2O AUTOML
# ============================================================
 
print("\n[7/7] H2O AutoML başlıyor...")
 
try:
    h2o.cluster().shutdown(prompt=False)
except Exception:
    pass
 
h2o.init(
    max_mem_size="4G",
    min_mem_size="1G",
    nthreads=4,
    port=54321,
    jvm_custom_args=["-XX:+UseG1GC", "-XX:MaxGCPauseMillis=200"]
)
 
# Feature sütunları — date, hedef ve ham AMOUNT dışındakiler
exclude_cols = ['date', 'AMOUNT', TARGET, LOG_TARGET]
feature_cols = [c for c in train.columns if c not in exclude_cols]
 
print(f"Feature sayısı : {len(feature_cols)}")
print(f"Features       : {feature_cols}")
 
# Tip dönüşümleri
cat_str_cols = ['BRANCH', 'CATEGORY_NAME1', 'REGION',
                'month_period'] if 'REGION' in train.columns else \
               ['BRANCH', 'CATEGORY_NAME1', 'month_period']
 
for col in cat_str_cols:
    if col in train.columns:
        train[col] = train[col].astype(str)
        test[col]  = test[col].astype(str)
 
for col in feature_cols:
    if col not in cat_str_cols and col in train.columns:
        train[col] = pd.to_numeric(train[col], errors='coerce').fillna(0).astype('float64')
        test[col]  = pd.to_numeric(test[col],  errors='coerce').fillna(0).astype('float64')
 
# Kronolojik val split (train'in son %15'i)
val_size = int(len(train) * 0.15)
val_df   = train.tail(val_size).copy()
train_df = train.iloc[:-val_size].copy()
 
print(f"train_df: {len(train_df):,} | val_df: {len(val_df):,} | test: {len(test):,}")
 
# H2OFrame
train_h2o = h2o.H2OFrame(train_df[feature_cols + [LOG_TARGET]])
val_h2o   = h2o.H2OFrame(val_df[feature_cols + [LOG_TARGET]])
test_h2o  = h2o.H2OFrame(test[feature_cols + [LOG_TARGET]])
 
# Tip zorlamaları
numeric_force = [c for c in feature_cols if c not in cat_str_cols]
for col in numeric_force:
    if col in train_h2o.columns:
        train_h2o[col] = train_h2o[col].asnumeric()
        val_h2o[col]   = val_h2o[col].asnumeric()
        test_h2o[col]  = test_h2o[col].asnumeric()
 
for col in cat_str_cols:
    if col in train_h2o.columns:
        train_h2o[col] = train_h2o[col].asfactor()
        val_h2o[col]   = val_h2o[col].asfactor()
        test_h2o[col]  = test_h2o[col].asfactor()
 
xgb_available = False
try:
    from h2o.estimators.xgboost import H2OXGBoostEstimator
    xgb_available = H2OXGBoostEstimator.available()
except Exception:
    pass
 
aml = H2OAutoML(
    max_models=15,
    max_runtime_secs=2400,
    seed=42,
    nfolds=0,
    sort_metric="RMSE",
    exclude_algos=["XGBoost", "DeepLearning"] if not xgb_available else ["DeepLearning"],
    verbosity=None
)
 
aml.train(
    x=feature_cols,
    y=LOG_TARGET,
    training_frame=train_h2o,
    validation_frame=val_h2o
)
 



[7/7] H2O AutoML başlıyor...
Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
  Java Version: openjdk version "25.0.1" 2025-10-21; OpenJDK Runtime Environment Homebrew (build 25.0.1); OpenJDK 64-Bit Server VM Homebrew (build 25.0.1, mixed mode, sharing)
  Starting server from /Users/semra/VsCodeProjects/5ml_task/food-101_classification/venv_ml/lib/python3.12/site-packages/h2o/backend/bin/h2o.jar
  Ice root: /var/folders/yp/pkv5qv7x6v11fn_9wtqqf68w0000gn/T/tmpog4dz21v
  JVM stdout: /var/folders/yp/pkv5qv7x6v11fn_9wtqqf68w0000gn/T/tmpog4dz21v/h2o_semra_started_from_python.out
  JVM stderr: /var/folders/yp/pkv5qv7x6v11fn_9wtqqf68w0000gn/T/tmpog4dz21v/h2o_semra_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Europe/Istanbul
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.10
H2O_cluster_version_age:,5 days
H2O_cluster_name:,H2O_from_python_semra_9fjn7k
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.984 Gb
H2O_cluster_total_cores:,10
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


Feature sayısı : 26
Features       : ['BRANCH', 'CATEGORY_NAME1', 'PRICE', 'REGION', 'dayofweek', 'month', 'weekofyear', 'is_weekend', 'day', 'is_monday', 'is_friday', 'month_period', 'lag_0', 'lag_1', 'lag_2', 'lag_7', 'lag_14', 'rolling_mean_3', 'rolling_mean_7', 'rolling_mean_14', 'rolling_std_7', 'weekly_trend', 'tomorrow_is_weekend', 'tomorrow_dayofweek', 'cat_avg_price', 'price_vs_cat']
train_df: 32,512 | val_df: 5,737 | test: 8,510
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Cannot build an XGBoost model - no backend found.
AutoML progress: |███████████████████████████████████████████████████████████████| (done) 100%


,number_of_trees,number_of_nternal_trees,model_size_in_bytes,min_depth,max_depth,mean_depth,min_leaves,max_leaves,mean_leaves
,50.0,50.0,8789764.0,20.0,20.0,20.0,12000.0,14189.0,13084.84
,timestamp,duration,number_of_trees,training_rmse,training_mae,training_deviance,validation_rmse,validation_mae,validation_deviance
,2026-03-17 12:44:15,0.016 sec,0.0,nan,nan,nan,nan,nan,nan
,2026-03-17 12:44:16,0.613 sec,5.0,0.7376678,0.5705089,0.5441538,0.6307709,0.4961094,0.3978720
,2026-03-17 12:44:16,1.077 sec,10.0,0.6727765,0.5256375,0.4526283,0.6058209,0.4776388,0.3670189
,2026-03-17 12:44:17,1.515 sec,15.0,0.6440996,0.5050569,0.4148643,0.5927950,0.4668388,0.3514060
,2026-03-17 12:44:17,1.962 sec,20.0,0.6282203,0.4940316,0.3946608,0.5892269,0.4640258,0.3471884
,2026-03-17 12:44:18,2.426 sec,25.0,0.6187994,0.4872634,0.3829127,0.5883692,0.4638234,0.3461784
,2026-03-17 12:44:18,2.881 sec,30.0,0.6123963,0.4824596,0.3750292,0.5877074,0.4633177,0.3453999
,2026-03-17 12:44:19,3.352 sec,35.0,0.6082133,0.4796862,0.3699234,0.5854397,0.4615446,0.3427396


In [23]:
# ============================================================
# SONUÇLAR
# ============================================================
 
print("\n" + "=" * 60)
print("SONUÇLAR — TEST SETİ (son 14 günün yarını tahmini):")
print("=" * 60)
 
best_model = aml.leader
 
pred_h2o   = best_model.predict(test_h2o)
y_pred_log = pred_h2o.as_data_frame()['predict'].values
y_pred     = np.clip(np.expm1(y_pred_log), 0, None)
y_test     = np.expm1(test[LOG_TARGET].values)
 
mae    = np.mean(np.abs(y_test - y_pred))
rmse   = np.sqrt(np.mean((y_test - y_pred) ** 2))
ss_res = np.sum((y_test - y_pred) ** 2)
ss_tot = np.sum((y_test - y_test.mean()) ** 2)
r2     = 1 - ss_res / ss_tot
 
# sMAPE — simetrik MAPE, küçük değerlere dayanıklı
# Formül: 2 * |gerçek - tahmin| / (|gerçek| + |tahmin|)
# 0-200% arasında, küçük değerlerde MAPE gibi patlamaz
smape = np.mean(
    2 * np.abs(y_test - y_pred) / (np.abs(y_test) + np.abs(y_pred) + 1e-8)
) * 100
 
# RMSSE — Root Mean Squared Scaled Error
# Naive modele (dünkü satış = yarınki tahmin) göre ne kadar iyi?
# < 1.0 → naive'den iyi, > 1.0 → naive'den kötü
naive_errors = np.abs(np.diff(y_test))  # |t - (t-1)|
naive_scale  = np.mean(naive_errors) if len(naive_errors) > 0 else 1.0
rmsse = rmse / (naive_scale + 1e-8)
 
# MdAE — Medyan Mutlak Hata (outlier'lara dayanıklı)
mdae = np.median(np.abs(y_test - y_pred))
 
# Yüzde kaçında 20'den az hata yapıyoruz?
within_20pct = np.mean(np.abs(y_test - y_pred) <= 0.20 * (y_test + 1e-8)) * 100
 
print(f"En iyi model  : {best_model.model_id}")
print(f"R²            : {r2:.4f}   → modelin açıklama gücü (1.0 mükemmel)")
print(f"MAE           : {mae:.2f}   → ortalama kaç birim yanıldık")
print(f"MdAE          : {mdae:.2f}   → medyan hata (outlier'a dayanıklı)")
print(f"RMSE          : {rmse:.2f}")
print(f"sMAPE         : {smape:.2f}%  → simetrik yüzde hata (MAPE'den güvenilir)")
print(f"RMSSE         : {rmsse:.4f}  → <1.0 ise naive'den iyi")
print(f"%20 içinde    : {within_20pct:.1f}%  → tahminlerin bu kadarı ±%%20 hata içinde")
 
print("\nLeaderboard:")
print(aml.leaderboard.head(rows=8))
 
try:
    importance = best_model.varimp(use_pandas=True)
    print("\nEn önemli 10 feature:")
    print(importance.head(10)[['variable', 'relative_importance', 'percentage']].to_string(index=False))
except Exception:
    importance = None
 
# ============================================================
# GÖRSELLEŞTİRME
# ============================================================
 
COLORS = {
    'bg'     : '#0f1117', 'panel': '#1a1d27',
    'accent1': '#7c6af7', 'accent2': '#f97316',
    'text'   : '#e2e8f0', 'subtext': '#94a3b8',
    'grid'   : '#2d3148', 'green': '#4ade80',
}
plt.rcParams.update({
    'figure.facecolor': COLORS['bg'], 'axes.facecolor': COLORS['panel'],
    'axes.edgecolor'  : COLORS['grid'], 'text.color'  : COLORS['text'],
    'axes.labelcolor' : COLORS['text'], 'xtick.color' : COLORS['subtext'],
    'ytick.color'     : COLORS['subtext'], 'grid.color': COLORS['grid'],
})
 
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor(COLORS['bg'])
 
# Plot 1: Gerçek vs Tahmin
sample = np.random.choice(len(y_test), min(2000, len(y_test)), replace=False)
axes[0].scatter(y_test[sample], y_pred[sample],
                alpha=0.4, color=COLORS['accent1'], s=10)
lim = max(y_test.max(), y_pred.max())
axes[0].plot([0, lim], [0, lim], 'r--', lw=1.5, label='y=x (mükemmel)')
axes[0].set_xlabel("Gerçek (Yarın)", fontsize=11)
axes[0].set_ylabel("Tahmin (Yarın)", fontsize=11)
axes[0].set_title(f"t+1 Tahmin\nR²={r2:.3f} | sMAPE={smape:.1f}%", fontsize=11, pad=12)
axes[0].legend(fontsize=9, facecolor=COLORS['panel'])
axes[0].grid(True, alpha=0.3)
 
# Plot 2: Kategori bazlı MAE
test_results = test[['date', 'BRANCH', 'CATEGORY_NAME1', TARGET]].copy()
test_results['pred'] = y_pred
test_results['mae']  = np.abs(test_results[TARGET] - test_results['pred'])
cat_mae = test_results.groupby('CATEGORY_NAME1')['mae'].mean().sort_values()
 
axes[1].barh(range(len(cat_mae)), cat_mae.values,
             color=COLORS['accent1'], alpha=0.85)
axes[1].set_yticks(range(len(cat_mae)))
axes[1].set_yticklabels(cat_mae.index, fontsize=8)
axes[1].set_xlabel("Ortalama MAE", fontsize=11)
axes[1].set_title("Kategori Bazlı Hata", fontsize=11, pad=12)
axes[1].grid(True, axis='x', alpha=0.3)
 
# Plot 3: Zaman serisi — en çok satan kombo
top_combo = test_results.groupby(['BRANCH', 'CATEGORY_NAME1'])[TARGET].sum().idxmax()
ts = test_results[
    (test_results['BRANCH'] == top_combo[0]) &
    (test_results['CATEGORY_NAME1'] == top_combo[1])
].sort_values('date')
 
if len(ts) > 0:
    axes[2].plot(range(len(ts)), ts[TARGET].values,
                 color=COLORS['green'], lw=2, marker='o', ms=4, label='Gerçek')
    axes[2].plot(range(len(ts)), ts['pred'].values,
                 color=COLORS['accent2'], lw=2, marker='s', ms=4,
                 linestyle='--', label='Tahmin (t+1)')
    axes[2].set_title(f"{top_combo[0][:15]}\n{top_combo[1]}", fontsize=9, pad=12)
    axes[2].set_xlabel("Gün", fontsize=11)
    axes[2].set_ylabel("Satış", fontsize=11)
    axes[2].legend(fontsize=9, facecolor=COLORS['panel'])
    axes[2].grid(True, alpha=0.3)
 
plt.tight_layout()
plt.savefig('results/plots/sales_forecast_v4.png', dpi=150,
            bbox_inches='tight', facecolor=COLORS['bg'])
plt.close()
print("\n✓ Grafik: results/plots/sales_forecast_v4.png")
 
# Model kaydet
save_dir = "./results/best_sales_model_v4"
os.makedirs(save_dir, exist_ok=True)
model_path = h2o.save_model(model=best_model, path=save_dir, force=True)
h2o.cluster().shutdown(prompt=False)
 
print("\n" + "=" * 60)
print("ÖZET:")
print("=" * 60)
print(f"Problem        : t+1 satış tahmini (bugünle yarını tahmin et)")
print(f"Aggregate      : Şube + Kategori + Gün")
print(f"Leakage        : YOK — tüm feature'lar t ve öncesi")
print(f"Split          : Kronolojik, son 14 gün test")
print(f"train_df       : {len(train_df):,} | val: {len(val_df):,} | test: {len(test):,}")
print(f"Feature sayısı : {len(feature_cols)}")
print(f"En iyi model   : {best_model.model_id}")
print(f"R²             : {r2:.4f}")
print(f"MAE            : {mae:.2f}")
print(f"MdAE           : {mdae:.2f}")
print(f"sMAPE          : {smape:.2f}%")
print(f"RMSSE          : {rmsse:.4f}  ({'✅ Naive''den iyi' if rmsse < 1 else '⚠️  Naive ile benzer'})")
print(f"%20 içinde     : {within_20pct:.1f}%")
print(f"Model          : {model_path}")
print("=" * 60)
print("✅ Tamamlandı!")



SONUÇLAR — TEST SETİ (son 14 günün yarını tahmini):
drf prediction progress: |███████████████████████████████████████████████████████| (done) 100%
En iyi model  : DRF_1_AutoML_1_20260317_124414
R²            : 0.6542   → modelin açıklama gücü (1.0 mükemmel)
MAE           : 6.40   → ortalama kaç birim yanıldık
MdAE          : 2.27   → medyan hata (outlier'a dayanıklı)
RMSE          : 22.58
sMAPE         : 55.91%  → simetrik yüzde hata (MAPE'den güvenilir)
RMSSE         : 2.4844  → <1.0 ise naive'den iyi
%20 içinde    : 21.4%  → tahminlerin bu kadarı ±%%20 hata içinde

Leaderboard:
model_id                                         rmse       mse       mae     rmsle    mean_residual_deviance
DRF_1_AutoML_1_20260317_124414               0.580803  0.337332  0.457982  0.203053                  0.337332
XRT_1_AutoML_1_20260317_124414               0.581019  0.337583  0.456014  0.202337                  0.337583
GBM_grid_1_AutoML_1_20260317_124414_model_5  0.594677  0.35364   0.465912  0.20612